# 06 · Catalogs and scaling

The earlier notebooks placed a few sources by hand. A real exposure has *thousands* per detector, read from a **catalog**, and the cost is dominated by dispersion. This notebook does two things: it shows the catalog-driven workflow, and it shows the one trick that makes thousands of sources tractable — **batched dispersion**, where a single compiled function loops over all sources instead of being re-called (and re-traced) per source.

This is mostly pipeline machinery rather than `roman_disperser` proper, but it's what you need at scale. Dispersion is compute-bound and embarrassingly parallel, so it's also where a **GPU** matters: we benchmark the scaling here on whatever backend you're running, and you get the GPU curve simply by re-running this same notebook on a GPU node.

> **Needs** the catalog hydrated (`pixi run hydrate --only catalog`). On a laptop CPU the scaling run below takes several minutes; on a GPU node it is far faster.

## 0 · Setup

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import time, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp
import zarr
import pyarrow.parquet as pq

from roman_disperser import paths, psf_model, star_disperser, galaxy_disperser, sersic, pipeline
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj

SCA = 5
ROOT = Path(os.environ.get("PIXI_PROJECT_ROOT", Path.cwd().parent))
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1 · Reading the catalog

The catalog is two files, and we only ever **read** them here:

- `metadata.parquet` — one row per source: position, `type` (`PSF` star / `SER` galaxy), morphology (`n`, `ba`, `pa`, `half_light_radius`), and two SED keys, `sed_index` and `flux_scale`;
- `seds.zarr` — the SEDs on a shared wavelength grid: a `star_seds` array and per-partition `galaxy_seds/sim_NNN` arrays.

The production loader lives in the disperser's `scripts/`, so we reproduce its few lines.

In [ ]:
cat = paths.catalog_dir()
meta = pq.read_table(cat / "metadata.parquet").to_pandas()
n_star = int((meta["type"] == "PSF").sum())
n_gal = int((meta["type"] == "SER").sum())
print(f"{len(meta):,} sources: {n_star:,} stars + {n_gal:,} galaxies")
print("columns:", list(meta.columns))

In [ ]:
store = zarr.open(str(cat / "seds.zarr"), mode="r")
wl_full = np.array(store["wavelengths"])                 # Angstroms
wl_mask = (wl_full >= 9000) & (wl_full <= 20000)         # trim to the grism band
wl_um = (wl_full[wl_mask] / 1e4).astype(np.float32)      # disperser wants microns
dlam_a = float(np.diff(wl_full[wl_mask]).mean())
print("zarr arrays:", list(store.keys()))
print(f"SED grid: {wl_um.size} samples, {dlam_a:.0f} Å spacing, {wl_um[0]:.2f}–{wl_um[-1]:.2f} µm")

## 2 · A source's spectrum

A source's SED is `seds[sed_index] × flux_scale` (in FLAM). For stars, `flux_scale` is the F158 flux; for galaxies it is 1.0 and the SED already carries the apparent flux. The `FLAM × sensitivity × Δλ` → counts conversion happens *inside* the disperser, so we only assemble FLAM here. As a sanity check, one star's SED:

In [ ]:
star_seds = np.array(store["star_seds"])[:, wl_mask].astype(np.float32)
row = meta[meta["type"] == "PSF"].iloc[0]
example = star_seds[int(row.sed_index)] * np.float32(row.flux_scale)

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(wl_um, example, color="C0")
ax.set(xlabel="wavelength [µm]", ylabel="FLAM", title=f"one star's SED (F158={row.F158:.3g})")
fig.tight_layout()

## Building your own catalog

The format is open, so you can build a catalog from your own sources and disperse it with the same code. The full spec — including the Zarr-v3 sharding that makes scattered-source reads fast — is in the disperser's `data/catalogs/README.md`; the essentials:

**`metadata.parquet`** — one row per source. The disperser reads:

| column | type | meaning |
|--------|------|---------|
| `ra`, `dec` | float64 | position (deg, ICRS) |
| `type` | str | `"PSF"` (point) or `"SER"` (Sérsic) |
| `n`, `half_light_radius`, `pa`, `ba` | float | Sérsic morphology (0/0/0/1 for points) |
| `sed_index` | int | row into the SED array |
| `flux_scale` | float | SED multiplier — `F158` (maggies) for stars, `1.0` for galaxies |
| `sim` | int | galaxy SED partition (0 for stars) |

Extra columns (`F158`, `z_obs`, …) are kept for analysis and ignored by the disperser.

**`seds.zarr`** — a shared `wavelengths` grid (Å, 2 Å spacing, covering 0.9–2.0 µm), a `star_seds` array, and per-partition `galaxy_seds/sim_NNN` arrays, all in **FLAM** (erg/s/cm²/Å). A source's spectrum is `seds[sed_index] × flux_scale`.

Minimal write (Zarr v3 — `shards` needs zarr-python ≥ 3):

```python
import pyarrow as pa, pyarrow.parquet as pq, zarr
from zarr.codecs import BloscCodec
comp = BloscCodec(cname="zstd", clevel=3, shuffle="shuffle")

pq.write_table(pa.table({"ra": ra, "dec": dec, "type": types, "sed_index": idx,
                         "flux_scale": fscale, "sim": sim, ...}), "metadata.parquet")

store = zarr.open("seds.zarr", mode="w")
store.create_array("wavelengths", data=wavelengths, compressors=comp)            # Angstroms
store.create_array("star_seds", data=star_templates, compressors=comp)            # (N_templ, N_wl) FLAM
N_wl = len(wavelengths); n_gal = len(galaxy_seds)
store.create_array("galaxy_seds/sim_001", data=galaxy_seds, compressors=comp,     # (N_gal, N_wl) FLAM
                   chunks=(10, N_wl), shards=(((n_gal + 9)//10)*10, N_wl))
```

The reference catalog is built from the Galacticus 4 deg² mock + a stellar atlas by `scripts/build_source_catalog.py` (`pixi run python scripts/build_source_catalog.py --sims 1-100`), and `scripts/verify_source_catalog.py` validates it.

## 3 · Selecting a field for a pointing

For a pointing (RA, Dec, PA) we keep sources in a cone, project them to the focal plane (`get_fpa_pos`), and keep those whose dispersed trace lands on our SCA (`select_sources_per_order`, wrapping `catalog.select_sources`). We point at the catalog centre.

> **Note (known limitation).** `get_fpa_pos` (sky→focal-plane) is currently only correct at **Dec = 0**; a fix is in progress. We point at the catalog centre (Dec = 0) here, so selection is valid. Off Dec = 0, place sources directly in SCA pixels (notebook 08) to avoid this path until the fix lands.

In [ ]:
PRA, PDEC, PPA = float(meta.ra.mean()), float(meta.dec.mean()), 0.0
model = RomanOpticalModel(config_file=str(paths.optical_model_path()))
opt = omj.make_sca_payload(model, sca=SCA, order="1")

cone = pipeline.cone_search(meta.ra.values, meta.dec.values, PRA, PDEC, 0.6)
mc = meta[cone].reset_index(drop=True)
xfpa, yfpa = omj.get_fpa_pos(jnp.array(mc.ra.values), jnp.array(mc.dec.values), PRA, PDEC, PPA)
_, any_mask = pipeline.select_sources_per_order({"1": opt}, xfpa, yfpa, orders=["1"])
any_mask = np.asarray(any_mask)
star_sel = any_mask & (mc["type"] == "PSF").values
gal_sel = any_mask & (mc["type"] == "SER").values
print(f"cone: {len(mc):,} sources;  on SCA{SCA}: {star_sel.sum()} stars + {gal_sel.sum()} galaxies")

In [ ]:
# assemble the field's star spectra (FLAM) and SCA pixel positions
star_meta = mc[star_sel]
star_spectra = star_seds[star_meta.sed_index.values] * star_meta.flux_scale.values.astype(np.float32)[:, None]
sx, sy = omj.fpa_to_sca(opt, xfpa[star_sel], yfpa[star_sel])
star_x, star_y = np.asarray(sx), np.asarray(sy)
print(f"{star_spectra.shape[0]} star spectra of length {star_spectra.shape[1]}")

## 4 · The batching trick: compile once, loop many

Dispersing N sources with a plain Python `for` loop is slow for two reasons: JAX re-traces whenever array shapes change, and there's per-call dispatch overhead. The pipeline avoids both by building **one** `jax.jit`-compiled function whose body is a `jax.lax.fori_loop` over sources:

- `make_batched_star_fori(disperser, sens, wavelengths, dlam)` returns a compiled `run(n, spectra, x, y, output)` that loops the disperser over `n` sources, doing the `FLAM × sens × Δλ` conversion inside;
- `disperse_batched_stars(run, spectra, x, y, output, batch_size)` feeds it the sources in **fixed-size, zero-padded** batches, passing the real count as a *traced* argument — so shapes never change and it compiles exactly once.

We disperse the **1st (science) order** only in this notebook, to keep the scaling demo clean. Orders 0 and 2 use the very same batched pattern, each with its own payload and sensitivity curve (notebook 02); the production pipeline co-adds all three.

Build it and disperse a small batch (the first call pays the one-time compile):

In [ ]:
psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order="1",
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
sens = pipeline.load_sensitivities(paths.sensitivity_dir(), SCA, wl_um)["1"]
disperser = star_disperser.make_star_disperser(psf, opt)
star_fori = pipeline.make_batched_star_fori(disperser, sens, jnp.asarray(wl_um), dlam_a)

N0 = min(50, star_spectra.shape[0])
t = time.time()
img = pipeline.disperse_batched_stars(star_fori, star_spectra[:N0], star_x[:N0], star_y[:N0],
                                      jnp.zeros((4088, 4088), jnp.float32), batch_size=1000)
img.block_until_ready()
img = np.asarray(img)
print(f"dispersed {N0} stars (incl. one-time compile) in {time.time()-t:.1f} s")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, origin="lower", cmap="inferno",
          norm=AsinhNorm(linear_width=img.max()*0.002, vmin=0, vmax=img.max()))
ax.set(title=f"{N0} catalog stars on SCA{SCA}", xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 5 · How it scales

The compiled function is now warm, so we time it on growing source counts — reusing the *same* `star_fori`, so no recompilation. Wall time grows linearly: each source is one independent `fori_loop` iteration.

In [ ]:
N_LIST = [n for n in [100, 250, 500] if n <= star_spectra.shape[0]]
times, field_stars = [], None
for n in N_LIST:
    out = jnp.zeros((4088, 4088), jnp.float32)
    t = time.time()
    out = pipeline.disperse_batched_stars(star_fori, star_spectra[:n], star_x[:n], star_y[:n], out, batch_size=1000)
    out.block_until_ready()
    times.append(time.time() - t)
    field_stars = np.asarray(out)          # keep the largest one for the field render below
    print(f"  {n:4d} stars: {times[-1]:6.1f} s")

backend = jax.default_backend()
results = {"backend": backend, "n": N_LIST, "time_s": times,
           "ms_per_source": [1e3 * t / n for t, n in zip(times, N_LIST)]}
# save the TIMING numbers (not a catalog) so a GPU re-run can be overlaid below
(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "outputs" / f"benchmark_{backend}.json").write_text(json.dumps(results, indent=2))
print(f"\n{backend}: {results['ms_per_source'][-1]:.0f} ms per source")

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.6))
a0.plot(N_LIST, times, "o-")
a0.set(xlabel="number of sources", ylabel="wall time [s]", title=f"scaling ({backend})")
a1.plot(N_LIST, results["ms_per_source"], "s-", color="C1")
a1.set(xlabel="number of sources", ylabel="ms per source",
       title="per-source cost (flat ⇒ linear scaling)")
fig.tight_layout()

## 6 · Galaxies in the field

Galaxies disperse with the *same* batched pattern, but with one extra ingredient: a galaxy is extended, so each one needs a **stamp** (a Sérsic image) in addition to its spectrum. The steps are:

1. pull each galaxy's SED from its `sim` partition in the zarr store;
2. render a Sérsic stamp per galaxy from the catalog morphology (`make_sersic_images`, with `catalog_r_eff_to_pixels` and `sky_pa_to_sca_theta`);
3. disperse them with `make_batched_galaxy_fori` / `disperse_batched_galaxies` — same `fori_loop`, plus a per-source `images` array (note the galaxy disperser's argument order is `(image, x, y, spectrum, wavelengths, output)`).

Galaxies cost more per source than stars (the stamp is warped and convolved at every wavelength), so we add a set of them to the star field for a realistic picture.

> **Density caveat.** The bundled catalog is a **1/100 sub-sample** (one `sim` partition), so only ~100 galaxies land on an SCA here — the true galaxy density is roughly **100× higher** (the full Galacticus mock has 100 such sub-samples). The field below is therefore sparse in galaxies; a full-density, all-orders run belongs on a GPU.

In [ ]:
gal_meta = mc[gal_sel].iloc[:50].reset_index(drop=True)
# 1. galaxy SEDs, grouped by sim partition
gal_spectra = np.zeros((len(gal_meta), wl_um.size), np.float32)
for sim, grp in gal_meta.groupby("sim"):
    arr = np.array(store[f"galaxy_seds/sim_{int(sim):03d}"])[:, wl_mask]
    for idx, r in grp.iterrows():
        gal_spectra[idx] = arr[int(r.sed_index)] * np.float32(r.flux_scale)
print(f"loaded {len(gal_meta)} galaxy SEDs")

In [ ]:
# 2. a Sérsic stamp per galaxy, from the catalog morphology
OV = psf["oversample"]
rpix = sersic.catalog_r_eff_to_pixels(gal_meta.half_light_radius.values, 0.11, oversample=OV)
theta = sersic.sky_pa_to_sca_theta(gal_meta.pa.values, PPA)
gal_imgs = np.asarray(sersic.make_sersic_images(rpix, gal_meta.n.values, gal_meta.ba.values, theta, 30*OV))
gal_imgs = gal_imgs / gal_imgs.sum(axis=(1, 2), keepdims=True)     # unit sum (flux is in the spectrum)
gx, gy = omj.fpa_to_sca(opt, xfpa[gal_sel][:50], yfpa[gal_sel][:50])
print("stamp array shape:", gal_imgs.shape)

In [ ]:
# 3. disperse the galaxies into the star field, same batched fori_loop
gal_disperser = galaxy_disperser.make_galaxy_disperser(psf, opt)
gal_fori = pipeline.make_batched_galaxy_fori(gal_disperser, sens, jnp.asarray(wl_um), dlam_a)
full = pipeline.disperse_batched_galaxies(gal_fori, gal_spectra, np.asarray(gx), np.asarray(gy),
                                          jnp.asarray(gal_imgs), jnp.asarray(field_stars), batch_size=64)
full = np.asarray(full)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(full, origin="lower", cmap="inferno",
          norm=AsinhNorm(linear_width=full.max()*0.001, vmin=0, vmax=full.max()))
ax.set(title=f"catalog field on SCA{SCA}: {N_LIST[-1]} stars + {len(gal_meta)} galaxies",
       xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 7 · The GPU curve — re-run this notebook

The dispersers are written for the GPU, where this benchmark runs far faster. There's no separate script: just run **this same notebook** on a GPU node (copy it, or open it with the `gpu` kernel) — the timing cell saves `outputs/benchmark_gpu.json`, and the cell below overlays whatever backends it finds.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for backend_name, style in [("cpu", "o-"), ("gpu", "s--")]:
    f = ROOT / "outputs" / f"benchmark_{backend_name}.json"
    if f.exists():
        d = json.loads(f.read_text())
        ax.plot(d["n"], d["time_s"], style, label=f"{backend_name} ({d['backend']})")
ax.set(xlabel="number of sources", ylabel="wall time [s]", title="dispersion scaling")
ax.legend()
fig.tight_layout()

ms = results["ms_per_source"][-1]
for n in (1e4, 1e5):
    print(f"{int(n):>7,} sources ≈ {ms/1e3*n/60:6.1f} min on this ({backend}) backend")
print("(One SCA of a deep field is ~10^4–10^5 sources; a survey is that × 18 SCAs × many pointings —")
print(" which is why the per-source cost, and the GPU, are what matter.)")

## 8 · The full pipeline at scale (in brief)

Producing a survey is the batched dispersion above wrapped in orchestration that lives in `scripts/build_grism_image.py` — worth knowing, not worth re-deriving here:

- **all 18 SCAs** per pointing, **many pointings** from an APT-derived `.ecsv`;
- **embarrassingly parallel** across `--worker-index / --num-workers` (one process per GPU);
- a persistent **JAX compilation cache** so workers compile once and reuse;
- per-SCA **FITS output** (`MODEL` count-rate + Poisson `ISIM`).

So the path from here is just *more of the same batched dispersion*, parallelised.

## Recap

- A field is **read** from a catalog (`metadata.parquet` + `seds.zarr`); a source's spectrum is `seds[sed_index] × flux_scale`.
- **Batched dispersion** (`make_batched_*_fori` + `disperse_batched_*`) compiles once and loops, so cost scales linearly with sources; galaxies add a per-source Sérsic stamp.
- The only thing this notebook *writes* is a small timing-results JSON — re-run on a GPU node to add the GPU curve.

**Next — [07 · JAX optical-model tools](07_jax_optical_model.ipynb).** A look under the hood at the differentiable optical model every disperser and the extractor are built on.